In [1]:
import os
os.chdir("/kpsort")
import cv2
from IPython.display import display, Image, clear_output
from tools.kpsort import Sort
from tools.loadpkl import *
from tools.AssignBeeHive import AssignBeeHive
from ultralytics import YOLO
import numpy as np
import pickle
import math
import time

In [2]:
"""
hive = AssignBeeHive("/kpsort/tools/out.png")
hive.gen_mask_w_sam(64, 3)
with open("hive.pkl", "wb") as f:
    pickle.dump(hive, f)
"""


'\nhive = AssignBeeHive("/kpsort/tools/out.png")\nhive.gen_mask_w_sam(64, 3)\nwith open("hive.pkl", "wb") as f:\n    pickle.dump(hive, f)\n'

In [3]:
with open("hive.pkl", "rb") as f:
    hive = pickle.load(f)

In [4]:
def imshow(img):
    _, buf = cv2.imencode(".jpg", img)
    display(Image(data=buf.tobytes()))
    clear_output(wait=True)
    
def check_overlap(individuals, threshold: int):
    fulls_sorted = list()
    fulls = dict()
    for i, individual in enumerate(individuals):
        if np.any(np.isnan(individual)):
            pass
        else:
            tmp = list()
            for j in range(0, len(individual) - 1, 2):
                tmp.append((individual[j], individual[j + 1]))
            tmp = sorted(tmp)
            fulls_sorted.append(np.append(np.array(tmp).flatten(), 0))
            fulls[tuple(np.append(np.array(tmp).flatten(), 0).tolist())] = i
    desirable2remove = list()
    for i, individual in enumerate(fulls_sorted):
        for j, ind in enumerate(fulls_sorted):
            if i >= j: continue
            oks_value = oks(individual, ind, 0.1)
            if oks_value > threshold:
                desirable2remove.append((fulls[tuple(individual)], fulls[tuple(ind)]))
    return np.array(list(desirable2remove))

In [5]:
MODE_SAVE = 0
MODE_SHOW = 1

mode = MODE_SHOW

path_csv = "sources/out_DLC_18fps/v18DLC_dlcrnetms5_bee1011_18Oct11shuffle1_200000_el.csv"
path_pkl = "sources/out_DLC_18fps/v18DLC_dlcrnetms5_bee1011_18Oct11shuffle1_200000_full.pickle"
with open(path_pkl, "rb") as file:
    data_pkl: dict = pickle.load(file)
data_csv = load_csv(path_csv)
color_map = iter(gen_random_colors(10000, 334))

model = YOLO("best2.pt", task="predict")
cap = cv2.VideoCapture("sources/v18.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
fourcc = cv2.VideoWriter_fourcc('m', 'p', '4', 'v')
video = cv2.VideoWriter("output/video4.mp4",fourcc, cap.get(cv2.CAP_PROP_FPS), size)

mot_tracker = Sort(iou_threshold=0.00001)

In [7]:
c = 0
colors = dict()
while True:
	st = time.time()
	#if c == 2: break
	success, frame = cap.read()
	if success:
		individuals, _ = assemble_w_yolo(model, frame, data_pkl, data_csv, 0.2, c)
		desirable2remove = check_overlap(individuals, 0.2)
		#desirable2remove = []
		for individual in individuals:
			for i in range(0, len(individual) - 1, 2):
				if math.isnan(individual[i]): continue
				cv2.circle(frame, (int(individual[i]), int(individual[i + 1])), 5, (0, 255, 0), 5)
		trackers = mot_tracker.update(individuals, desirable2remove)
		#print(trackers)
		for d in trackers:
			d = d.astype(np.int32)
			#if np.any(d < 0): continue
			mask = str(bin(int(d[6])))[2:].zfill(6)
			if d[-1] not in colors:
				colors[d[-1]] = next(color_map)
			for i in range(0, len(d), 2):
				if i > 4: break
				if mask[i] != "1" and i != 1:
					cv2.circle(frame, (d[i], d[i + 1]), 5, colors[d[-1]], 5)
					#cv2.drawMarker(frame, (d[i], d[i + 1]), colors[d[7]])
				if i == 4:
					cv2.circle(frame, (d[i], d[i + 1]), 5, colors[d[-1]], 5)
					#cv2.putText(frame, f"@{hive.pos2id((d[i], d[i + 1]))}", (d[0], d[1]), cv2.FONT_HERSHEY_PLAIN, 5, colors[d[-1]], 1, cv2.LINE_AA)
			cv2.putText(frame, str(d[-1]), (d[0], d[1]), cv2.FONT_HERSHEY_PLAIN, 5, colors[d[-1]], 1, cv2.LINE_AA)
		c += 1
		imshow(frame)
		#elapsed_time = time.time() - st
		#if elapsed_time < 1 / fps:
		#	time.sleep(1 / fps - elapsed_time)
		#time.sleep(0.1)
	else: 
		break

KeyboardInterrupt: 